# Lab: Cross-Validation and the Bootstrap
### Based on ISLP Chapter 5 — Resampling Methods

This notebook walks through the two big resampling techniques used to estimate how well a model will perform on new data, and how uncertain a statistic is:

1. **Cross-validation** (validation set, LOOCV, k-fold) — estimates a model's **test error**.
2. **The bootstrap** — estimates the **standard error / variability** of almost any statistic, by resampling the data itself.

We'll use the `Auto` dataset (predicting `mpg` from `horsepower`) for the cross-validation part, and the `Portfolio` dataset for the bootstrap part.

## 0. Setup

First, load the packages we'll need throughout the notebook.

In [1]:
import numpy as np
import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                          summarize,
                          poly)
from sklearn.model_selection import train_test_split

In [2]:
# A few extra imports specific to cross-validation and the bootstrap
from functools import partial
from sklearn.model_selection import (cross_validate,
                                      KFold,
                                      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm

---
## 1. The Validation Set Approach

**Idea:** randomly split the data in half. Fit the model on one half (training set), then measure its error on the other half (validation set). This gives a quick, honest estimate of test error — but it depends heavily on *which* random split you happened to draw, and it wastes half the data for training.

We'll estimate the test MSE of predicting `mpg` from `horsepower` in the `Auto` dataset.

In [3]:
Auto = load_data('Auto')

# Split into training / validation sets of equal size (196 / 196)
# random_state fixes the seed so results are reproducible
Auto_train, Auto_valid = train_test_split(Auto,
                                           test_size=196,
                                           random_state=0)

Fit a simple linear regression of `mpg` on `horsepower` using only the training half:

In [4]:
hp_mm = MS(['horsepower'])
X_train = hp_mm.fit_transform(Auto_train)
y_train = Auto_train['mpg']

model = sm.OLS(y_train, X_train)
results = model.fit()

Now evaluate it on the held-out validation set to get the validation MSE:

In [5]:
X_valid = hp_mm.transform(Auto_valid)
y_valid = Auto_valid['mpg']
valid_pred = results.predict(X_valid)

validation_mse = np.mean((y_valid - valid_pred)**2)
validation_mse

np.float64(23.61661706966988)

**Result:** validation MSE ≈ **23.62** for the linear fit.

Let's also check whether a quadratic or cubic fit does better. We write a small helper `evalMSE()` that fits a model on a training set and returns its MSE on a test set.

In [6]:
def evalMSE(terms, response, train, test):
    mm = MS(terms)
    X_train = mm.fit_transform(train)
    y_train = train[response]

    X_test = mm.transform(test)
    y_test = test[response]

    results = sm.OLS(y_train, X_train).fit()
    test_pred = results.predict(X_test)

    return np.mean((y_test - test_pred)**2)

In [7]:
MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                        'mpg',
                        Auto_train,
                        Auto_valid)
MSE

array([23.61661707, 18.76303135, 18.79694163])

These are the validation MSEs for **linear, quadratic, and cubic** fits: `23.62, 18.76, 18.80`. The jump from linear → quadratic is a big improvement; cubic adds essentially nothing.

**Caveat:** because the validation set approach depends on one random split, a *different* split can give noticeably different numbers. Let's re-run with a different `random_state` to see this in action.

In [8]:
Auto_train, Auto_valid = train_test_split(Auto,
                                           test_size=196,
                                           random_state=3)

MSE = np.zeros(3)
for idx, degree in enumerate(range(1, 4)):
    MSE[idx] = evalMSE([poly('horsepower', degree)],
                        'mpg',
                        Auto_train,
                        Auto_valid)
MSE

array([20.75540796, 16.94510676, 16.97437833])

New split → new MSEs: `20.76, 16.95, 16.97`. The absolute numbers moved quite a bit, but the *conclusion* is the same both times: quadratic beats linear, and cubic adds nothing. This instability is exactly why cross-validation (next section) is usually preferred over a single validation split.

---
## 2. Cross-Validation

Cross-validation fixes the "one lucky/unlucky split" problem by repeatedly splitting the data, training on most of it, and validating on the rest — then averaging the error across all the splits.

`statsmodels` (which we used above) doesn't have built-in cross-validation tools, but `sklearn` does. The `ISLP` package provides a small wrapper, `sklearn_sm()`, that lets us cross-validate a `statsmodels` model using `sklearn`'s cross-validation functions.

### 2a. Leave-One-Out Cross-Validation (LOOCV)

LOOCV is the extreme case: each "fold" leaves out just **one** observation, trains on all the rest, and tests on that one point — repeated for every observation in the dataset.

In [9]:
hp_model = sklearn_sm(sm.OLS, MS(['horsepower']))
X, Y = Auto.drop(columns=['mpg']), Auto['mpg']

cv_results = cross_validate(hp_model,
                             X, Y,
                             cv=Auto.shape[0])   # cv = n -> LOOCV

cv_err = np.mean(cv_results['test_score'])
cv_err

np.float64(24.23151351792922)

LOOCV estimate of test MSE for the linear fit ≈ **24.23**.

Let's repeat this for polynomial fits of degree 1 through 5, to see where test error levels off:

In [10]:
cv_error = np.zeros(5)
H = np.array(Auto['horsepower'])
M = sklearn_sm(sm.OLS)

for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    M_CV = cross_validate(M, X, Y, cv=Auto.shape[0])
    cv_error[i] = np.mean(M_CV['test_score'])

cv_error

array([24.23151352, 19.24821312, 19.33498406, 19.42443031, 19.03321178])

`[24.23, 19.25, 19.34, 19.42, 19.03]` — same story as before: a sharp drop from linear to quadratic, then the error basically flattens out. Higher-degree polynomials aren't buying us anything.

*(Side note: `np.power.outer(H, np.arange(d+1))` raises every element of `H` to every power in `0..d` and lays the results out as columns — a quick way to build a polynomial design matrix.)*

### 2b. k-Fold Cross-Validation

LOOCV is accurate but can be slow (n model fits!). **k-fold CV** instead splits the data into k roughly equal chunks (commonly k=10), trains on k−1 of them, and validates on the one left out — cycling through all k chunks. It's much faster and, in practice, works just as well.

In [11]:
cv_error = np.zeros(5)
cv = KFold(n_splits=10,
           shuffle=True,
           random_state=0)  # fixed folds, reused for every polynomial degree

for i, d in enumerate(range(1, 6)):
    X = np.power.outer(H, np.arange(d + 1))
    M_CV = cross_validate(M, X, Y, cv=cv)
    cv_error[i] = np.mean(M_CV['test_score'])

cv_error

array([24.20766449, 19.18533142, 19.27626666, 19.47848402, 19.13720959])

10-fold CV gives `[24.21, 19.19, 19.28, 19.48, 19.14]` — nearly identical to LOOCV, but computed much faster. Same conclusion: quadratic is enough.

### 2c. Cross-validation with a custom splitter (`ShuffleSplit`)

`cross_validate()` accepts any splitting strategy. `ShuffleSplit` lets us replicate the plain validation-set approach (one random split), or repeat it many times to see how much the estimate itself varies.

In [12]:
# A single 196/196 split - same as the validation-set approach from Part 1
validation = ShuffleSplit(n_splits=1,
                           test_size=196,
                           random_state=0)

results = cross_validate(hp_model,
                          Auto.drop(['mpg'], axis=1),
                          Auto['mpg'],
                          cv=validation)
results['test_score']

array([23.61661707])

In [13]:
# Repeat the random split 10 times to see the variability in the estimate
validation = ShuffleSplit(n_splits=10,
                           test_size=196,
                           random_state=0)

results = cross_validate(hp_model,
                          Auto.drop(['mpg'], axis=1),
                          Auto['mpg'],
                          cv=validation)

results['test_score'].mean(), results['test_score'].std()

(np.float64(23.802232661034168), np.float64(1.4218450941091842))

Mean ≈ 23.80, std ≈ 1.42. This spread isn't a rigorous standard error (the random splits overlap, so the scores aren't independent), but it gives a rough sense of how much a single validation-set estimate can bounce around just from the luck of the split.

---
## 3. The Bootstrap

The bootstrap answers a different question than cross-validation: instead of "how well will my model predict new data?", it asks **"how uncertain is this statistic / estimate?"** — and it does so *without* needing a mathematical formula for the standard error. You just resample your own data (with replacement) thousands of times and see how much the statistic wobbles.

### 3a. Warm-up example: estimating the standard error of α

We use the `Portfolio` dataset. The goal is to estimate a parameter α (the optimal fraction of money to invest in one asset vs. another to minimize risk) and find out how uncertain that estimate is.

In [14]:
Portfolio = load_data('Portfolio')

def alpha_func(D, idx):
    """Estimate of alpha using only the observations in idx."""
    cov_ = np.cov(D[['X', 'Y']].loc[idx], rowvar=False)
    return ((cov_[1, 1] - cov_[0, 1]) /
            (cov_[0, 0] + cov_[1, 1] - 2 * cov_[0, 1]))

In [15]:
# Estimate of alpha using ALL 100 observations
alpha_func(Portfolio, range(100))

np.float64(0.57583207459283)

Now let's create **one** bootstrap sample — 100 observations drawn *with replacement* from the original 100 — and recompute α on it:

In [16]:
rng = np.random.default_rng(0)
alpha_func(Portfolio,
           rng.choice(100, 100, replace=True))

np.float64(0.6074452469619004)

Notice this gives a slightly different α than using the full data — that's the whole point. If we do this thousands of times and look at the spread of the resulting α estimates, that spread *is* the standard error.

Let's turn that into a reusable function, `boot_SE()`, that bootstraps **any** statistic-computing function `B` times and returns its estimated standard error:

In [17]:
def boot_SE(func, D, n=None, B=1000, seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]

    for _ in range(B):
        idx = rng.choice(D.index, n, replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2

    return np.sqrt(second_ / B - (first_ / B)**2)

`_` is just a throwaway loop variable — we don't care about its value, only that the loop runs `B` times. Now estimate SE(α) using 1,000 bootstrap resamples:

In [18]:
alpha_SE = boot_SE(alpha_func, Portfolio, B=1000, seed=0)
alpha_SE

np.float64(0.09118176521277699)

Bootstrap estimate: **SE(α̂) ≈ 0.091**.

### 3b. Estimating the accuracy of linear regression coefficients

The bootstrap is especially useful for checking regression coefficient standard errors, since the textbook SE formulas rely on assumptions (e.g. correct linear model, fixed predictors, known noise variance) that don't always hold. Here we bootstrap the coefficients of `mpg ~ horsepower`.

First, a generic function that refits an OLS model on a bootstrapped set of rows:

In [19]:
def boot_OLS(model_matrix, response, D, idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_, X_).fit().params

`boot_SE()` expects a function of just `(D, idx)`, but `boot_OLS()` also needs the model formula and response column. We use `functools.partial` to "freeze" those first two arguments, leaving a function of `(D, idx)` only:

In [20]:
hp_func = partial(boot_OLS, MS(['horsepower']), 'mpg')

Sanity check — run it on 10 bootstrap samples by hand to see the intercept/slope estimates jump around a bit each time.

*(Note: newer versions of the `ISLP` package index `Auto` by car name rather than by row number, so we sample from `Auto.index` directly instead of hardcoding `392` — this keeps the code correct no matter how the DataFrame is indexed.)*

In [21]:
rng = np.random.default_rng(0)
# Use Auto.index (rather than a hardcoded 392) so this works
# regardless of how the DataFrame happens to be indexed.
np.array([hp_func(Auto, rng.choice(Auto.index, Auto.shape[0], replace=True))
          for _ in range(10)])

array([[39.12226577, -0.1555926 ],
       [37.18648613, -0.13915813],
       [37.46989244, -0.14112749],
       [38.56723252, -0.14830116],
       [38.95495707, -0.15315141],
       [39.12563927, -0.15261044],
       [38.45763251, -0.14767251],
       [38.43372587, -0.15019447],
       [37.87581142, -0.1409544 ],
       [37.95949036, -0.1451333 ]])

Now do it properly with 1,000 bootstrap replications to get stable standard error estimates:

In [22]:
hp_se = boot_SE(hp_func, Auto, B=1000, seed=10)
hp_se

intercept     0.731176
horsepower    0.006092
dtype: float64

Compare these to the textbook formula-based standard errors from `sm.OLS` (via `summarize`):

In [23]:
hp_model.fit(Auto, Auto['mpg'])
model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

Compare the two outputs above: in the original textbook run, the **bootstrap SEs** come out noticeably **larger** than the **formula-based SEs**. (The exact numbers you get can shift slightly depending on package version / random seed behavior, but the direction of the effect is the interesting part.)

This isn't a flaw in the bootstrap — it's the opposite. The classical formula assumes the linear model is exactly correct and that all the noise variance σ² is estimated from residuals of that (possibly wrong) model. Since `mpg` vs `horsepower` is actually *curved*, a straight-line fit has inflated residuals, which inflates the formula's σ̂² and biases its SEs. The bootstrap makes no such assumption, so it's likely giving the more trustworthy answer here.

As a check: if we bootstrap a **quadratic** fit instead (which matches the data's true shape much better), the two approaches should agree much more closely.

In [24]:
quad_model = MS([poly('horsepower', 2, raw=True)])
quad_func = partial(boot_OLS, quad_model, 'mpg')

boot_SE(quad_func, Auto, B=1000)

intercept                                  1.538641
poly(horsepower, degree=2, raw=True)[0]    0.024696
poly(horsepower, degree=2, raw=True)[1]    0.000090
dtype: float64

In [25]:
M = sm.OLS(Auto['mpg'], quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64

Indeed — for the quadratic model, the bootstrap SEs and the formula-based SEs line up much more closely, confirming the explanation above.

---
## Summary

| Technique | Answers | Key functions |
|---|---|---|
| Validation set | Quick test-error estimate, one split | `train_test_split` |
| LOOCV | Test-error estimate, leaves out 1 obs at a time | `cross_validate(..., cv=n)` |
| k-Fold CV | Test-error estimate, leaves out 1/k of data at a time | `cross_validate(..., cv=KFold(...))` |
| Bootstrap | Standard error / uncertainty of *any* statistic | custom `boot_SE()` with resampling |

**Takeaways to remember for a quiz:**
- Validation-set estimates are fast but *noisy* — they depend heavily on the random split.
- LOOCV has (almost) no randomness but can be computationally expensive for large n.
- k-fold CV (typically k=5 or 10) is the usual practical compromise: nearly as accurate as LOOCV, much faster.
- The bootstrap resamples the **data itself** (with replacement) to estimate the variability of a statistic — no formula required, and no assumptions about the model being "correct."
- Bootstrap SEs can legitimately *disagree* with textbook formula SEs when the model's assumptions (e.g. correct functional form) are violated — and when that happens, the bootstrap is usually the one to trust more.